# 03 · Anatomy of a Front-Page Post

What separates a 500-point story from one that gets 3 points and dies? We analyse title features, timing, and metadata to find the real predictors.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.loader import db
from src.nlp import title_features, extract_domain
from src.viz import set_style, heatmap_2d, bar_chart, save

set_style()
con = db()

In [2]:
stories = con.execute("""
    SELECT
        title,
        url,
        score,
        comment_count,
        posted_at,
        HOUR(posted_at)      AS hour_utc,
        DAYOFWEEK(posted_at) AS dow,
        YEAR(posted_at)      AS year
    FROM stories
    WHERE score >= 3
      AND year BETWEEN 2015 AND 2024
""").df()

stories = title_features(stories)
stories['domain'] = stories['url'].apply(extract_domain)
print(f'{len(stories):,} stories (score >= 3, 2015–2024)')


1,297,362 stories (score >= 3, 2015–2024)


## When to post: score by hour × day-of-week

In [3]:
pivot = stories.groupby(['dow', 'hour_utc'])['score'].mean().unstack()
pivot.index = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

fig = heatmap_2d(
    pivot,
    title='Mean story score by day-of-week and hour (UTC), 2015–2024, score ≥ 3',
    xlabel='Hour (UTC)',
    ylabel='Day of week',
)
save(fig, '../data/fig_timing_heatmap.png')
plt.show()

best_slot = pivot.stack().idxmax()
print(f'Best posting slot: {best_slot[0]} at {best_slot[1]:02d}:00 UTC (mean score)')


Best posting slot: Sun at 14:00 UTC (mean score)


## Title length vs score

In [4]:
bins = pd.cut(stories['title_len_words'], bins=[0,5,8,11,14,17,30], labels=['1–5','6–8','9–11','12–14','15–17','18+'])
length_score = stories.groupby(bins, observed=True)['score'].mean().reset_index()
length_score.columns = ['title_words', 'mean_score']
display(length_score)

fig = bar_chart(
    list(length_score['title_words'].astype(str)),
    list(length_score['mean_score']),
    title='Mean score by title word count (score ≥ 3, 2015–2024)',
    xlabel='Mean score',
    horizontal=True,
)
save(fig, '../data/fig_title_length.png')
plt.show()


,title_words,mean_score
0,1–5,49.497912
1,6–8,40.142824
2,9–11,36.530608
3,12–14,35.719096
4,15–17,35.668534
5,18+,36.981912


## Questions vs statements vs numbers

In [5]:
feature_scores = {
    'Question (ends with ?)': stories[stories['title_is_question']]['score'].mean(),
    'Contains a number':      stories[stories['title_has_number']]['score'].mean(),
    'Positive sentiment':     stories[stories['title_sentiment'] > 0.1]['score'].mean(),
    'Negative sentiment':     stories[stories['title_sentiment'] < -0.1]['score'].mean(),
    'Neutral sentiment':      stories[stories['title_sentiment'].between(-0.1, 0.1)]['score'].mean(),
    'All stories (baseline)': stories['score'].mean(),
}

fig = bar_chart(
    list(feature_scores.keys()),
    list(feature_scores.values()),
    title='Mean score by title feature (score ≥ 3, 2015–2024)',
    xlabel='Mean score',
)
save(fig, '../data/fig_title_features.png')
plt.show()

print('\nMean score by title feature:')
print(pd.Series(feature_scores).sort_values(ascending=False).to_string())



Mean score by title feature:
Contains a number         45.371823
Neutral sentiment         41.142009
Negative sentiment        40.945160
All stories (baseline)    40.153645
Positive sentiment        36.527742
Question (ends with ?)    29.046668


## Discussion maximizers: high comments, low score

In [6]:
controversy = stories[(stories['score'] >= 2) & stories['comment_count'].notna()].copy()
controversy['controversy_ratio'] = controversy['comment_count'] / (controversy['score'] + 1)

print('Top 20 discussion-maximizing posts (most comments relative to score):')
top_controversial = controversy.nlargest(20, 'controversy_ratio')[['title', 'year', 'score', 'comment_count', 'controversy_ratio']]
display(top_controversial)

Top 20 discussion-maximizing posts (most comments relative to score):


,title,year,score,comment_count,controversy_ratio
901801,The Demise of the Mildly Dynamic Website,2022,4,91,18.2
602500,Ask HN: Interesting sci-fi movies BY TOPIC?,2020,3,57,14.25
1076446,Jeffrey Sachs: the West’s false narrative abou...,2023,6,71,10.142857
784812,Some states push to ban “discrimination” again...,2021,23,236,9.833333
1047839,Ask HN: I've run Linux for 13 years. Is it tim...,2023,3,39,9.75
1267993,Russian Family Lived Alone in the Siberian Wil...,2024,4,48,9.6
1269256,Play Free OvO Unblocked Game Online,2024,5,57,9.5
1111880,California's Governor Vetoes State Ban on Driv...,2023,14,142,9.466667
224883,Warning Get Ready for Regulations That Restric...,2016,4,45,9.0
1121009,Automakers Have Big Hopes for EVs; Buyers Aren...,2023,13,121,8.642857
